# Deep dive 7/13: `transactions`

First FACT table in this pass, and the biggest table yet (dictionary declares 5,000,000 rows) -- extra care here per your instruction: every column gets its own check, Bronze and Silver, no full-table dump (not feasible at this size), aggregates and samples instead.

This is also the most logic-heavy Silver build so far. From `transactions_spec` in `table_specs.py`:

- **`amount_usd` has a two-layer fallback.** Bronze's own `amount_usd` is null for a documented 57.34% of rows. Silver fills the gap with `amount * that day's exact-date FX rate` (joined via `dim_fx_rates` on `currency` + `transaction_date`'s date part), falling back to `amount` unchanged when `currency = 'USD'`. Unlike `products` (which uses the *latest* available rate, `v_fx_latest_to_usd`), `transactions` joins the *exact-date* rate, since a transaction has a real date to join against.
- **`amount_usd_is_estimated`** is a Silver-only derived boolean flagging which rows got the fallback vs. which had a genuine Bronze-sourced `amount_usd`. Sections 8-9 verify both the fallback math and this flag's logic directly, not from memory.
- Three FKs (`customer_id`, `product_id`, `branch_id`) -- reproduced directly in section 6, since this project's rule going forward is to never trust a prior notebook's FK numbers without re-running them here.

Dictionary NOT NULL columns: `transaction_id`, `transaction_date`, `process_date`, `product_id`, `customer_id`, `transaction_type`, `amount`, `currency`, `channel`, `transaction_country`, `transaction_status`, `is_fraud`. `contracts.py`'s current `required` tuple is missing `transaction_country` -- section 3 checks it explicitly as a fix candidate. Nullable: `transaction_category`, `amount_usd`, `branch_id`, `merchant_name`, `merchant_category`, `transaction_city`, `response_code`, `fraud_score`, `latitude`, `longitude`.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side

5,000,000 rows -- no full dump. 20-row sample instead, same transaction_ids in both so you can compare the exact transform.

In [2]:
sample_ids = q("SELECT transaction_id FROM bronze.transactions ORDER BY transaction_id LIMIT 20")["transaction_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

print("Bronze:")
display(q_full(f"SELECT * FROM bronze.transactions WHERE transaction_id IN ({ids_sql}) ORDER BY transaction_id"))

Bronze:


,transaction_id,transaction_date,process_date,product_id,customer_id,transaction_type,transaction_category,amount,currency,amount_usd,channel,branch_id,merchant_name,merchant_category,transaction_country,transaction_city,transaction_status,response_code,is_fraud,fraud_score,latitude,longitude,_source_file,_ingested_at,_source_table,day,month,year
0,TRX-00009BG2YZ08GS40IPEB,2024-10-30 16:48:27,2024-10-30,PRD-K5NWYB5WUIJI,CLI-4TM97TE1LKJC,Transfer,NaN,25680715.84,COP,6420.18,POS,NaN,NaN,NaN,Colombia,Cartagena,Approved,00,False,26.8,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=10/da...,2026-09-28 19:23:02.809394-03:00,transactions,30,10,2024
1,TRX-0000TMHW2ZLRUQGBOQFQ,2025-09-05 23:47:14,2025-09-05,PRD-GV4Z1E6DZ4N9,CLI-IQRUI56ACGQ3,Transfer,NaN,9127.11,USD,NaN,ATM,SUC-39R34AY6,NaN,NaN,Argentina,Buenos Aires,Approved,00,False,NaN,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2025/month=09/da...,2026-09-28 19:23:02.809394-03:00,transactions,05,09,2025
2,TRX-000114DXH9IPLEJDOBJ1,2023-11-10 04:54:48,2023-11-09,PRD-FRTEM0I6KC3G,CLI-51K9CTVFZP3K,Purchase,Services,659183.12,COP,164.8,POS,NaN,Cable TV,Services,Colombia,NaN,Approved,00,False,24.72,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2023/month=11/da...,2026-09-28 19:23:02.809394-03:00,transactions,09,11,2023
3,TRX-00017JR2QTRY4O649YLT,2024-10-02 14:09:49,2024-10-02,PRD-OWKLFQTAIXKT,CLI-VJAC40SS27WD,Transfer,NaN,15942677.78,COP,NaN,Branch,NaN,NaN,NaN,Colombia,Cali,Approved,00,False,21.16,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=10/da...,2026-09-28 19:23:02.809394-03:00,transactions,02,10,2024
4,TRX-0001D19BIMBIVTOSNO9I,2025-07-30 19:18:40,2025-07-30,PRD-RPKUQH3NQQ7A,CLI-VNUX4NFOQXOW,Withdrawal,NaN,360.55,USD,NaN,POS,NaN,NaN,NaN,Argentina,NaN,Approved,NaN,False,17.85,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2025/month=07/da...,2026-09-28 19:23:02.809394-03:00,transactions,30,07,2025
5,TRX-0001DMU984SDIPZOTNYJ,2024-11-04 10:37:08,2024-11-04,PRD-TZ6YW9AP65DW,CLI-K78MEF4VDK7P,Withdrawal,NaN,127377.79,ARS,363.94,POS,NaN,NaN,NaN,Argentina,Mendoza,Approved,00,False,NaN,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=11/da...,2026-09-28 19:23:02.809394-03:00,transactions,04,11,2024
6,TRX-0001EXUQ0XWT4W62B99A,2025-12-27 10:13:21,2025-12-27,PRD-OQ1KTL7N6LMD,CLI-HBQWV3L8TBI3,Purchase,Food,291.35,USD,NaN,POS,NaN,NaN,Food,México,Ciudad de México,Approved,00,False,23.28,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2025/month=12/da...,2026-09-28 19:23:02.809394-03:00,transactions,27,12,2025
7,TRX-0001OXQTJPPXH55C3TRL,2024-09-15 23:14:23,2024-09-15,PRD-QZF3KAMQZZB0,CLI-JL3OQXH6FRWI,Deposit,NaN,12524287.07,COP,3131.07,App,NaN,NaN,NaN,Colombia,Bogotá,Approved,00,False,2.36,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=09/da...,2026-09-28 19:23:02.809394-03:00,transactions,15,09,2024
8,TRX-00025892DKUTCIIES2D6,2024-02-20 05:09:48,2024-02-19,PRD-9LVZKSBIW9EB,CLI-9WS78NP80RPL,Deposit,NaN,2773.41,USD,NaN,Web,NaN,NaN,NaN,México,Guadalajara,Approved,00,False,15.7,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=02/da...,2026-09-28 19:23:02.809394-03:00,transactions,19,02,2024
9,TRX-0002751RBEKEJR0KM5RY,2024-02-22 21:01:44,2024-02-22,PRD-E70GURSJHML3,CLI-JNOGSV2TXXJ2,Payment,Food,2519439.15,COP,629.86,App,NaN,NaN,NaN,Colombia,Cartagena,Approved,00,False,21.08,NaN,NaN,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/transactions/year=2024/month=02/da...,2026-09-28 19:23:02.809394-03:00,transactions,22,02,2024


In [3]:
print("Silver:")
display(q_full(f"SELECT * FROM silver.fact_transactions WHERE transaction_id IN ({ids_sql}) ORDER BY transaction_id"))

Silver:


,transaction_id,transaction_date,process_date,product_id,customer_id,transaction_type,transaction_category,amount,currency,amount_usd,amount_usd_is_estimated,channel,branch_id,merchant_name,merchant_category,transaction_country,transaction_city,transaction_status,response_code,is_fraud,fraud_score,latitude,longitude
0,TRX-00009BG2YZ08GS40IPEB,2024-10-30 16:48:27,2024-10-30,PRD-K5NWYB5WUIJI,CLI-4TM97TE1LKJC,Transfer,NaN,25680715.84,COP,6420.180000,False,POS,NaN,NaN,NaN,Colombia,Cartagena,Approved,00,False,26.80,NaN,NaN
1,TRX-0000TMHW2ZLRUQGBOQFQ,2025-09-05 23:47:14,2025-09-05,PRD-GV4Z1E6DZ4N9,CLI-IQRUI56ACGQ3,Transfer,NaN,9127.11,USD,9127.110000,False,ATM,SUC-39R34AY6,NaN,NaN,Argentina,Buenos Aires,Approved,00,False,NaN,NaN,NaN
2,TRX-000114DXH9IPLEJDOBJ1,2023-11-10 04:54:48,2023-11-09,PRD-FRTEM0I6KC3G,CLI-51K9CTVFZP3K,Purchase,Services,659183.12,COP,164.800000,False,POS,NaN,Cable TV,Services,Colombia,NaN,Approved,00,False,24.72,NaN,NaN
3,TRX-00017JR2QTRY4O649YLT,2024-10-02 14:09:49,2024-10-02,PRD-OWKLFQTAIXKT,CLI-VJAC40SS27WD,Transfer,NaN,15942677.78,COP,4065.382834,True,Branch,NaN,NaN,NaN,Colombia,Cali,Approved,00,False,21.16,NaN,NaN
4,TRX-0001D19BIMBIVTOSNO9I,2025-07-30 19:18:40,2025-07-30,PRD-RPKUQH3NQQ7A,CLI-VNUX4NFOQXOW,Withdrawal,NaN,360.55,USD,360.550000,False,POS,NaN,NaN,NaN,Argentina,NaN,Approved,NaN,False,17.85,NaN,NaN
5,TRX-0001DMU984SDIPZOTNYJ,2024-11-04 10:37:08,2024-11-04,PRD-TZ6YW9AP65DW,CLI-K78MEF4VDK7P,Withdrawal,NaN,127377.79,ARS,363.940000,False,POS,NaN,NaN,NaN,Argentina,Mendoza,Approved,00,False,NaN,NaN,NaN
6,TRX-0001EXUQ0XWT4W62B99A,2025-12-27 10:13:21,2025-12-27,PRD-OQ1KTL7N6LMD,CLI-HBQWV3L8TBI3,Purchase,Food,291.35,USD,291.350000,False,POS,NaN,NaN,Food,México,Ciudad de México,Approved,00,False,23.28,NaN,NaN
7,TRX-0001OXQTJPPXH55C3TRL,2024-09-15 23:14:23,2024-09-15,PRD-QZF3KAMQZZB0,CLI-JL3OQXH6FRWI,Deposit,NaN,12524287.07,COP,3131.070000,False,App,NaN,NaN,NaN,Colombia,Bogotá,Approved,00,False,2.36,NaN,NaN
8,TRX-00025892DKUTCIIES2D6,2024-02-20 05:09:48,2024-02-19,PRD-9LVZKSBIW9EB,CLI-9WS78NP80RPL,Deposit,NaN,2773.41,USD,2773.410000,False,Web,NaN,NaN,NaN,México,Guadalajara,Approved,00,False,15.70,NaN,NaN
9,TRX-0002751RBEKEJR0KM5RY,2024-02-22 21:01:44,2024-02-22,PRD-E70GURSJHML3,CLI-JNOGSV2TXXJ2,Payment,Food,2519439.15,COP,629.860000,False,App,NaN,NaN,NaN,Colombia,Cartagena,Approved,00,False,21.08,NaN,NaN


## 1. Column inventory -- names AND types, Bronze vs. Silver

In [4]:
expected_silver_columns = [
    "transaction_id", "transaction_date", "process_date", "product_id", "customer_id",
    "transaction_type", "transaction_category", "amount", "currency", "amount_usd",
    "amount_usd_is_estimated", "channel", "branch_id", "merchant_name", "merchant_category",
    "transaction_country", "transaction_city", "transaction_status", "response_code",
    "is_fraud", "fraud_score", "latitude", "longitude",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_transactions'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_transactions:", missing or "none")
print("Extra in silver.fact_transactions:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.fact_transactions: none
Extra in silver.fact_transactions: none

Column order as built: ['transaction_id', 'transaction_date', 'process_date', 'product_id', 'customer_id', 'transaction_type', 'transaction_category', 'amount', 'currency', 'amount_usd', 'amount_usd_is_estimated', 'channel', 'branch_id', 'merchant_name', 'merchant_category', 'transaction_country', 'transaction_city', 'transaction_status', 'response_code', 'is_fraud', 'fraud_score', 'latitude', 'longitude']


In [5]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='transactions' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_transactions' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,transaction_id,VARCHAR
1,transaction_date,VARCHAR
2,process_date,VARCHAR
3,product_id,VARCHAR
4,customer_id,VARCHAR
5,transaction_type,VARCHAR
6,transaction_category,VARCHAR
7,amount,VARCHAR
8,currency,VARCHAR
9,amount_usd,VARCHAR


Silver columns + types:


,column_name,data_type
0,transaction_id,VARCHAR
1,transaction_date,TIMESTAMP
2,process_date,DATE
3,product_id,VARCHAR
4,customer_id,VARCHAR
5,transaction_type,VARCHAR
6,transaction_category,VARCHAR
7,amount,DOUBLE
8,currency,VARCHAR
9,amount_usd,DOUBLE


## 2. Row-count reconciliation + dedup

In [6]:
bronze_total = q("SELECT count(*) AS n FROM bronze.transactions")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT transaction_id) AS n FROM bronze.transactions")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_transactions")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT transaction_id) AS n FROM silver.fact_transactions")["n"][0]

print(f"bronze.transactions:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct transaction_id")
print(f"silver.fact_transactions: {silver_total:,} rows, {silver_distinct_pk:,} distinct transaction_id")
print()
print("Dictionary declares 5,000,000 rows -- matches" if bronze_total == 5_000_000 else f"Dictionary declares 5,000,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.transactions:      4,425,008 rows, 4,425,008 distinct transaction_id
silver.fact_transactions: 4,425,008 rows, 4,425,008 distinct transaction_id

Dictionary declares 5,000,000 rows -- MISMATCH, Bronze has 4,425,008
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 12 dictionary-declared columns

Including `transaction_country`, which the dictionary marks NOT NULL but `contracts.py`'s `required` tuple currently omits -- checking it here as a fix candidate.

In [7]:
declared_not_null = [
    "transaction_id", "transaction_date", "process_date", "product_id", "customer_id",
    "transaction_type", "amount", "currency", "channel", "transaction_country",
    "transaction_status", "is_fraud",
]

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.transactions
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_transactions WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL",
                 "in_contracts_required": col in ("transaction_id","transaction_date","process_date","product_id","customer_id","transaction_type","amount","currency","channel","transaction_status","is_fraud"),
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,column,declared,in_contracts_required,bronze_nulls,silver_nulls,cast_introduced_nulls
0,transaction_id,NOT NULL,True,0,0,0
1,transaction_date,NOT NULL,True,0,0,0
2,process_date,NOT NULL,True,0,0,0
3,product_id,NOT NULL,True,0,0,0
4,customer_id,NOT NULL,True,0,0,0
5,transaction_type,NOT NULL,True,0,0,0
6,amount,NOT NULL,True,0,0,0
7,currency,NOT NULL,True,0,0,0
8,channel,NOT NULL,True,0,0,0
9,transaction_country,NOT NULL,False,0,0,0


## 4. Nullable columns -- population rate, Bronze vs. Silver

`transaction_category`, `amount_usd`, `branch_id`, `merchant_name`, `merchant_category`, `transaction_city`, `response_code`, `fraud_score`, `latitude`, `longitude`. (`amount_usd`'s *pre-fallback* Bronze rate is checked here; the post-fallback Silver rate is checked separately in section 8, since Silver deliberately fills most of the gap.)

In [8]:
nullable_cols = ["transaction_category", "amount_usd", "branch_id", "merchant_name",
                  "merchant_category", "transaction_city", "response_code", "fraud_score",
                  "latitude", "longitude"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.transactions")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.transactions
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_transactions WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

,column,total,bronze_populated,bronze_pct,silver_populated,silver_pct
0,transaction_category,4425008,1731488,39.13,1731488,39.13
1,amount_usd,4425008,1887552,42.66,4424973,100.00
2,branch_id,4425008,1387932,31.37,1387932,31.37
3,merchant_name,4425008,1029234,23.26,1029234,23.26
4,merchant_category,4425008,1028793,23.25,1028793,23.25
5,transaction_city,4425008,3982397,90.00,3982397,90.00
6,response_code,4425008,4203975,95.00,4203975,95.00
7,fraud_score,4425008,3539851,80.00,3539851,80.00
8,latitude,4425008,857328,19.37,857328,19.37
9,longitude,4425008,857293,19.37,857293,19.37


## 5. Domain checks -- `transaction_type`, `transaction_category`, `channel`, `transaction_status`

None of these four are currently gated in `contracts.py`.

In [9]:
# transaction_type -- dictionary: Deposit, Withdrawal, Transfer, Payment, Purchase, Advance
print("Bronze:")
display(q("SELECT transaction_type, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT transaction_type, count(*) AS n FROM silver.fact_transactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,transaction_type,n
0,Purchase,1083406
1,Withdrawal,964673
2,Transfer,896438
3,Payment,738964
4,Deposit,609409
5,Adjustment,132118


Silver:


,transaction_type,n
0,Purchase,1083406
1,Withdrawal,964673
2,Transfer,896438
3,Payment,738964
4,Deposit,609409
5,Adjustment,132118


In [10]:
# transaction_category -- dictionary: Food, Transport, Services, Entertainment, Health, Other
print("Bronze:")
display(q("SELECT transaction_category, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT transaction_category, count(*) AS n FROM silver.fact_transactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,transaction_category,n
0,NaN,2693520
1,Food,432468
2,Services,345370
3,Other,260518
4,Transport,260316
5,Entertainment,259643
6,Health,173173


Silver:


,transaction_category,n
0,NaN,2693520
1,Food,432468
2,Services,345370
3,Other,260518
4,Transport,260316
5,Entertainment,259643
6,Health,173173


In [11]:
# channel -- dictionary: ATM, Branch, Web, App, POS, Transfer
print("Bronze:")
display(q("SELECT channel, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT channel, count(*) AS n FROM silver.fact_transactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,channel,n
0,POS,1548161
1,ATM,1328334
2,Web,663445
3,App,663414
4,Branch,132495
5,Transfer,89159


Silver:


,channel,n
0,POS,1548161
1,ATM,1328334
2,Web,663445
3,App,663414
4,Branch,132495
5,Transfer,89159


In [12]:
# transaction_status -- dictionary: Approved, Declined, Pending, Reversed
print("Bronze:")
display(q("SELECT transaction_status, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT transaction_status, count(*) AS n FROM silver.fact_transactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,transaction_status,n
0,Approved,4070681
1,Declined,221234
2,Pending,88343
3,Reversed,44750


Silver:


,transaction_status,n
0,Approved,4070681
1,Declined,221234
2,Pending,88343
3,Reversed,44750


## 6. FK integrity -- reproduced directly for all three FKs, not trusted from memory

In [13]:
fk_checks = [
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("product_id", "silver.dim_products", "product_id"),
    ("branch_id", "silver.dim_branches", "branch_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_transactions WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_transactions t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

,fk,populated,orphans,orphan_pct
0,customer_id -> silver.dim_customers.customer_id,4425008,0,0.0
1,product_id -> silver.dim_products.product_id,4425008,0,0.0
2,branch_id -> silver.dim_branches.branch_id,1387932,0,0.0


## 7. `amount`/`currency` -- cast correctness, range sanity, currency domain

In [14]:
q("""
    SELECT
        typeof(amount) AS silver_type,
        min(amount) AS min_amount, max(amount) AS max_amount,
        count(*) FILTER (WHERE amount < 0) AS negative_amounts,
        count(*) FILTER (WHERE amount = 0) AS zero_amounts
    FROM silver.fact_transactions
    GROUP BY ALL
""")

,silver_type,min_amount,max_amount,negative_amounts,zero_amounts
0,DOUBLE,5.0,39999828.48,0,0


In [15]:
# Is a negative amount (if any exist) concentrated in a particular transaction_type (e.g.
# Withdrawal modeled as negative), or is it noise?
q("""
    SELECT transaction_type, count(*) AS n, min(amount) AS min_amt, max(amount) AS max_amt
    FROM silver.fact_transactions
    WHERE amount <= 0
    GROUP BY 1
    ORDER BY n DESC
""")

,transaction_type,n,min_amt,max_amt


In [16]:
# currency -- dictionary's 4-currency set (MXN, COP, ARS, USD), same as daily_exchange_rates/products
declared_currencies = {"MXN", "COP", "ARS", "USD"}
real_currencies = set(q("SELECT DISTINCT currency AS v FROM bronze.transactions")["v"])
print("currency values NOT in the declared 4-currency set:", real_currencies - declared_currencies or "none -- full match")
display(q("SELECT currency, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))

currency values NOT in the declared 4-currency set: none -- full match


,currency,n
0,USD,2437979
1,COP,1194444
2,ARS,792585


## 8. `amount_usd` fallback logic -- confirm the documented 57.34% Bronze null rate, and verify Silver's fill

In [17]:
bronze_amount_usd_nulls = q("SELECT count(*) AS n FROM bronze.transactions WHERE amount_usd IS NULL OR trim(amount_usd) = ''")["n"][0]
total = q("SELECT count(*) AS n FROM bronze.transactions")["n"][0]
print(f"Bronze amount_usd null rate: {bronze_amount_usd_nulls:,}/{total:,} = {100*bronze_amount_usd_nulls/total:.2f}%")
print("(table_specs.py's comment states 57.34% from the original Bronze profile -- confirming against live data)")

silver_amount_usd_nulls = q("SELECT count(*) AS n FROM silver.fact_transactions WHERE amount_usd IS NULL")["n"][0]
print(f"\nSilver amount_usd null rate AFTER the fallback: {silver_amount_usd_nulls:,}/{total:,} = {100*silver_amount_usd_nulls/total:.2f}%")
print("(should be far lower -- only rows where BOTH Bronze's amount_usd AND the FX join are unavailable remain null)")

Bronze amount_usd null rate: 2,537,456/4,425,008 = 57.34%
(table_specs.py's comment states 57.34% from the original Bronze profile -- confirming against live data)

Silver amount_usd null rate AFTER the fallback: 35/4,425,008 = 0.00%
(should be far lower -- only rows where BOTH Bronze's amount_usd AND the FX join are unavailable remain null)


In [18]:
# If any Silver amount_usd rows are still null, what's the common cause -- no FX rate for that
# (currency, date) combination? (e.g. a date outside dim_fx_rates' 2023-06-17/2026-06-17 coverage,
# or a currency not in the FX table.)
q("""
    SELECT t.currency, count(*) AS n,
           min(TRY_CAST(t.transaction_date AS DATE)) AS earliest, max(TRY_CAST(t.transaction_date AS DATE)) AS latest
    FROM silver.fact_transactions t
    WHERE t.amount_usd IS NULL
    GROUP BY 1
    ORDER BY n DESC
""")

,currency,n,earliest,latest
0,ARS,20,2026-06-18,2026-06-18
1,COP,15,2026-06-18,2026-06-18


## 9. `amount_usd_is_estimated` -- reproduce the flag's logic by hand and diff against Silver

In [19]:
# Reproduce the exact expression from table_specs.py against Bronze + a manual FX join, then
# compare row-for-row against what Silver actually stored.
reproduced = q("""
    SELECT t.transaction_id,
           (TRY_CAST(t.amount_usd AS DOUBLE) IS NULL
            AND UPPER(TRIM(t.currency)) <> 'USD'
            AND TRY_CAST(t.amount AS DOUBLE) * fx.exchange_rate IS NOT NULL) AS expected_flag
    FROM bronze.transactions t
    LEFT JOIN silver.dim_fx_rates fx
      ON fx.source_currency = trim(t.currency)
     AND fx.target_currency = 'USD'
     AND fx.rate_date = TRY_CAST(t.transaction_date AS DATE)
""")
actual = q("SELECT transaction_id, amount_usd_is_estimated AS actual_flag FROM silver.fact_transactions")

merged = reproduced.merge(actual, on="transaction_id")
mismatches = merged[merged["expected_flag"] != merged["actual_flag"]]
print(f"Rows where the reproduced flag disagrees with Silver's stored amount_usd_is_estimated: {len(mismatches):,} / {len(merged):,}")
if len(mismatches):
    display(mismatches.head(20))

print()
print("How many rows were actually flagged as estimated, and what share of all transactions is that?")
display(q("""
    SELECT amount_usd_is_estimated, count(*) AS n, round(100.0*count(*)/(SELECT count(*) FROM silver.fact_transactions), 2) AS pct
    FROM silver.fact_transactions GROUP BY 1 ORDER BY 1
"""))

Rows where the reproduced flag disagrees with Silver's stored amount_usd_is_estimated: 0 / 4,425,008

How many rows were actually flagged as estimated, and what share of all transactions is that?


,amount_usd_is_estimated,n,pct
0,False,4325566,97.75
1,True,99442,2.25


## 10. `transaction_date`/`process_date` -- cast correctness, internal consistency, window check

In [20]:
q("""
    SELECT
        typeof(transaction_date) AS txn_date_type, typeof(process_date) AS process_date_type,
        count(*) FILTER (WHERE transaction_date IS NULL) AS txn_date_nulls,
        count(*) FILTER (WHERE process_date IS NULL) AS process_date_nulls,
        min(transaction_date) AS earliest_txn, max(transaction_date) AS latest_txn,
        min(process_date) AS earliest_process, max(process_date) AS latest_process
    FROM silver.fact_transactions
    GROUP BY ALL
""")

,txn_date_type,process_date_type,txn_date_nulls,process_date_nulls,earliest_txn,latest_txn,earliest_process,latest_process
0,TIMESTAMP,DATE,0,0,2023-06-17 06:01:30,2026-06-18 05:59:41,2023-06-17,2026-06-17


In [21]:
# process_date is documented as the partition key -- does it actually match the date part of
# transaction_date, or can a transaction be processed on a different day than it happened?
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE process_date = CAST(transaction_date AS DATE)) AS process_date_matches_txn_date,
        count(*) FILTER (WHERE process_date <> CAST(transaction_date AS DATE)) AS process_date_differs,
        count(*) FILTER (WHERE process_date < CAST(transaction_date AS DATE)) AS processed_before_txn,
        count(*) FILTER (WHERE process_date > CAST(transaction_date AS DATE)) AS processed_after_txn
    FROM silver.fact_transactions
""")

,total,process_date_matches_txn_date,process_date_differs,processed_before_txn,processed_after_txn
0,4425008,3318701,1106307,1106307,0


In [22]:
# Same window check as every other table.
q("""
    SELECT
        count(*) FILTER (WHERE transaction_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE transaction_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_transactions
""")

,before_window,after_window
0,0,1352


## 11. `is_fraud`/`fraud_score` -- boolean cast, range sanity, cross-field consistency

In [23]:
q("""
    SELECT
        typeof(is_fraud) AS is_fraud_type,
        count(*) FILTER (WHERE is_fraud IS NULL) AS is_fraud_nulls,
        count(*) FILTER (WHERE is_fraud) AS fraud_true, count(*) AS total,
        round(100.0 * count(*) FILTER (WHERE is_fraud) / count(*), 4) AS fraud_rate_pct
    FROM silver.fact_transactions
    GROUP BY ALL
""")

,is_fraud_type,is_fraud_nulls,fraud_true,total,fraud_rate_pct
0,BOOLEAN,0,4316,4425008,0.0975


In [24]:
q("""
    SELECT
        typeof(fraud_score) AS silver_type,
        min(fraud_score) AS min_score, max(fraud_score) AS max_score,
        count(*) FILTER (WHERE fraud_score < 0 OR fraud_score > 100) AS out_of_0_100_range,
        count(*) FILTER (WHERE fraud_score IS NOT NULL) AS populated, count(*) AS total
    FROM silver.fact_transactions
    GROUP BY ALL
""")

,silver_type,min_score,max_score,out_of_0_100_range,populated,total
0,DOUBLE,0.0,99.99,0,3539851,4425008


In [25]:
# Does a higher fraud_score actually line up with is_fraud = true? Not a strict validity rule
# (a score is a risk signal, not a guarantee), but worth seeing the shape.
q("""
    SELECT is_fraud, count(*) AS n,
           round(avg(fraud_score), 2) AS avg_fraud_score,
           round(min(fraud_score), 2) AS min_fraud_score, round(max(fraud_score), 2) AS max_fraud_score
    FROM silver.fact_transactions
    WHERE fraud_score IS NOT NULL
    GROUP BY 1
""")

,is_fraud,n,avg_fraud_score,min_fraud_score,max_fraud_score
0,False,3536426,15.00,0.00,30.00
1,True,3425,49.46,0.01,99.99


## 12. `transaction_country` -- canonicalization coverage

In [26]:
print("Bronze raw:")
display(q("SELECT transaction_country, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))
print("Silver (post-canonicalization):")
display(q("SELECT transaction_country, count(*) AS n FROM silver.fact_transactions GROUP BY 1 ORDER BY n DESC"))

Bronze raw:


,transaction_country,n
0,México,2105794
1,Colombia,1289503
2,Argentina,867561
3,USA,40621
4,Spain,40542
5,Mexico,40515
6,Brazil,40472


Silver (post-canonicalization):


,transaction_country,n
0,México,2146309
1,Colombia,1289503
2,Argentina,867561
3,USA,40621
4,Spain,40542
5,Brazil,40472


## 13. `latitude`/`longitude` -- range sanity, and a direct check for the same near-zero corruption found in `branches`

`branches` had 167/350 rows (47.7%) with both coordinates corrupted to tiny near-zero decimals. Worth checking whether `transactions` -- an entirely separate source system per the dictionary (Core Banking, not Internal) -- shows the same pattern or not.

In [27]:
q("""
    SELECT
        typeof(latitude) AS lat_type, typeof(longitude) AS lon_type,
        count(*) FILTER (WHERE latitude IS NOT NULL) AS lat_populated,
        count(*) FILTER (WHERE longitude IS NOT NULL) AS lon_populated,
        count(*) AS total,
        min(latitude) AS min_lat, max(latitude) AS max_lat,
        min(longitude) AS min_lon, max(longitude) AS max_lon
    FROM silver.fact_transactions
    GROUP BY ALL
""")

,lat_type,lon_type,lat_populated,lon_populated,total,min_lat,max_lat,min_lon,max_lon
0,DOUBLE,DOUBLE,857328,857293,4425008,-35.603698,5.710998,-75.072097,0.99999


In [28]:
# Same near-(0,0) diagnostic used on branches: real LATAM coordinates should never be this close
# to the Gulf of Guinea point (0, 0).
q("""
    SELECT
        count(*) FILTER (WHERE latitude IS NOT NULL AND longitude IS NOT NULL
                          AND abs(latitude) < 0.2 AND abs(longitude) < 0.2) AS near_zero_zero,
        count(*) FILTER (WHERE latitude IS NOT NULL AND longitude IS NOT NULL) AS both_populated
    FROM silver.fact_transactions
""")

,near_zero_zero,both_populated
0,16481,814415


## 14. `merchant_name`/`merchant_category`/`response_code` -- free text / low-cardinality spot checks

In [29]:
total = q("SELECT count(*) AS n FROM bronze.transactions")["n"][0]
print(f"merchant_name populated: {q('SELECT count(*) AS n FROM bronze.transactions WHERE merchant_name IS NOT NULL')['n'][0]:,} / {total:,}")
print("merchant_category (Bronze), top 20:")
display(q("SELECT merchant_category, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC LIMIT 20"))
print("response_code (Bronze):")
display(q("SELECT response_code, count(*) AS n FROM bronze.transactions GROUP BY 1 ORDER BY n DESC"))

merchant_name populated: 1,029,234 / 4,425,008
merchant_category (Bronze), top 20:


,merchant_category,n
0,NaN,3396215
1,Food,256846
2,Services,205124
3,Other,155029
4,Transport,154931
5,Entertainment,153960
6,Health,102903


response_code (Bronze):


,response_code,n
0,00,3867312
1,NaN,221033
2,14,84472
3,51,84179
4,05,84141
5,54,83871


In [30]:
# Is merchant_name/merchant_category populated only for the transaction_types where it makes
# sense (e.g. Purchase), or spread across all types regardless?
q("""
    SELECT transaction_type,
           count(*) AS n,
           count(*) FILTER (WHERE merchant_name IS NOT NULL) AS with_merchant_name,
           round(100.0 * count(*) FILTER (WHERE merchant_name IS NOT NULL) / count(*), 1) AS pct_with_merchant
    FROM bronze.transactions
    GROUP BY 1
    ORDER BY n DESC
""")

,transaction_type,n,with_merchant_name,pct_with_merchant
0,Purchase,1083406,1029234,95.0
1,Withdrawal,964673,0,0.0
2,Transfer,896438,0,0.0
3,Payment,738964,0,0.0
4,Deposit,609409,0,0.0
5,Adjustment,132118,0,0.0


## 15. Follow-up A -- `currency` never includes MXN, despite México being the largest `transaction_country`

Section 7 found `currency` only ever takes 3 of the 4 declared values (USD, COP, ARS) -- **MXN never appears once in 4,425,008 rows**, even though `transaction_country = 'México'` accounts for ~2.1M rows (the single largest country). Before writing this up, checking whether `currency` tracks `transaction_country` at all, or something else (e.g. the owning product's currency, or a currency chosen independently of geography).

In [31]:
# Does currency correlate with transaction_country at all?
q("""
    SELECT transaction_country, currency, count(*) AS n
    FROM silver.fact_transactions
    GROUP BY 1, 2
    ORDER BY 1, n DESC
""")

,transaction_country,currency,n
0,Argentina,ARS,752794
1,Argentina,USD,102708
2,Argentina,COP,12059
3,Brazil,USD,20726
4,Brazil,COP,11887
5,Brazil,ARS,7859
6,Colombia,COP,1134801
7,Colombia,USD,146719
8,Colombia,ARS,7983
9,México,USD,2126409


In [32]:
# Cross-check against the owning product's own currency -- does a transaction's currency always
# match its product's currency, or can they differ (e.g. a MXN-denominated product transacted in USD)?
q("""
    SELECT p.currency AS product_currency, t.currency AS transaction_currency, count(*) AS n
    FROM silver.fact_transactions t
    JOIN silver.dim_products p ON p.product_id = t.product_id
    GROUP BY 1, 2
    ORDER BY 1, n DESC
""")

,product_currency,transaction_currency,n
0,ARS,ARS,792585
1,COP,COP,1194444
2,USD,USD,2437979


In [33]:
# Does products itself ever have a MXN-currency product at all? If products has zero MXN too,
# this isn't a transactions-specific gap -- it traces back further.
q("SELECT currency, count(*) AS n FROM silver.dim_products GROUP BY 1 ORDER BY n DESC")

,currency,n
0,USD,220501
1,COP,107975
2,ARS,71524


## 16. Follow-up B -- 25% of rows have `process_date` one calendar day before `transaction_date`'s date part

Section 10 found 1,106,307 rows (25%) where `process_date < CAST(transaction_date AS DATE)`, always by exactly one day (never more, never in the other direction). Every example row shown in section 0's sample had an early-morning `transaction_date` (e.g. 04:54, 05:09, 01:20, 03:00, 00:21) rolling back to the previous day's `process_date`. Checking whether this is fully explained by time-of-day -- consistent with a timezone-offset artifact between whatever generated `transaction_date` (apparently local time) and whatever generated `process_date` (apparently a different, earlier-shifted zone) -- rather than being scattered/random.

In [34]:
# Hour-of-day distribution for rows where process_date rolled back vs. rows where it matched.
q("""
    SELECT
        (process_date < CAST(transaction_date AS DATE)) AS process_date_rolled_back,
        EXTRACT(hour FROM transaction_date) AS txn_hour,
        count(*) AS n
    FROM silver.fact_transactions
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

,process_date_rolled_back,txn_hour,n
0,False,6,184017
1,False,7,184301
2,False,8,183747
3,False,9,184764
4,False,10,183991
5,False,11,184169
6,False,12,184227
7,False,13,184648
8,False,14,184612
9,False,15,184479


In [35]:
# If the rollback is a clean timezone-offset artifact, EVERY rolled-back row should have an
# early transaction_date hour (e.g. before some fixed cutoff), and no rolled-back row should have
# a late-day hour. Check the max/min hour on each side directly instead of eyeballing the table above.
q("""
    SELECT
        (process_date < CAST(transaction_date AS DATE)) AS process_date_rolled_back,
        min(EXTRACT(hour FROM transaction_date)) AS min_hour,
        max(EXTRACT(hour FROM transaction_date)) AS max_hour,
        count(*) AS n
    FROM silver.fact_transactions
    GROUP BY 1
""")

,process_date_rolled_back,min_hour,max_hour,n
0,True,0,6,1106307
1,False,6,23,3318701


## Checklist

- [ ] Column inventory (1): all 22 Silver columns present (21 typed + `amount_usd_is_estimated`), types correct
- [ ] Row reconciliation (2): 5,000,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 12 declared columns populated -- note whether `transaction_country` is really clean (fix candidate for `contracts.py`'s `required`)
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `transaction_type`/`transaction_category`/`channel`/`transaction_status` match declared sets
- [ ] FK integrity (6): `customer_id`/`product_id`/`branch_id` orphan rates reproduced directly
- [ ] `amount`/`currency` (7): no negative amounts (or a sensible explanation if there are), currency domain clean
- [ ] `amount_usd` fallback (8): Bronze's 57.34% null rate confirmed, Silver's post-fallback rate much lower
- [ ] `amount_usd_is_estimated` (9): flag logic reproduces exactly by hand
- [ ] Dates (10): `process_date` vs. `transaction_date` consistency, window check
- [ ] `is_fraud`/`fraud_score` (11): fraud rate, score range, cross-field shape
- [ ] `transaction_country` (12): full canonicalization coverage
- [ ] `latitude`/`longitude` (13): note if the same branches-style near-zero corruption shows up here too
- [ ] `merchant_name`/`merchant_category`/`response_code` (14): population patterns make sense

## Your own exploration

Free cells below -- `con`, `q()`, and `q_full()` are already set up.

In [36]:
con.close()
print("closed")

closed
